In [3]:
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace("\n", " ", regex=False)
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )
    return df

satcat_raw = pd.read_csv("../data/raw/satcat.csv")
ucs_raw = pd.read_excel("../data/raw/ucs_satellite_database_2023.xlsx")
owid_raw = pd.read_csv("../data/raw/owid_objects_launched.csv")

satcat = clean_columns(satcat_raw)
ucs = clean_columns(ucs_raw)
owid = clean_columns(owid_raw)

for name, df in [("SATCAT", satcat), ("UCS", ucs), ("OWID", owid)]:
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print("shape:", df.shape)
    print("\ncolumnas:")
    print(df.columns.tolist())
    print("\nprimeras filas:")
    print(df.head(3))


SATCAT
shape: (68590, 17)

columnas:
['object_name', 'object_id', 'norad_cat_id', 'object_type', 'ops_status_code', 'owner', 'launch_date', 'launch_site', 'decay_date', 'period', 'inclination', 'apogee', 'perigee', 'rcs', 'data_status_code', 'orbit_center', 'orbit_type']

primeras filas:
  object_name  object_id  norad_cat_id object_type ops_status_code owner launch_date launch_site  decay_date  period  inclination  apogee  perigee    rcs data_status_code orbit_center orbit_type
0    SL-1 R/B  1957-001A             1         R/B               D   CIS  1957-10-04       TYMSC  1957-12-01   96.19        65.10   938.0    214.0  20.42              NaN           EA        IMP
1   SPUTNIK 1  1957-001B             2         PAY               D   CIS  1957-10-04       TYMSC  1958-01-03   96.10        65.00  1080.0     64.0    NaN              NaN           EA        IMP
2   SPUTNIK 2  1957-002A             3         PAY               D   CIS  1957-11-03       TYMSC  1958-04-14  103.74        6

In [2]:
def find_cols(df, keywords):
    cols = df.columns.tolist()
    return [c for c in cols if any(k in c for k in keywords)]

print("SATCAT posibles claves:", find_cols(satcat, ["norad", "object", "launch", "owner", "apogee", "perigee"]))
print("UCS posibles claves:", find_cols(ucs, ["norad", "cospar", "operator", "owner", "purpose", "mass", "power", "launch"]))
print("OWID posibles claves:", find_cols(owid, ["entity", "code", "year", "object", "launch"]))

SATCAT posibles claves: ['object_name', 'object_id', 'norad_cat_id', 'object_type', 'owner', 'launch_date', 'launch_site', 'apogee', 'perigee']
UCS posibles claves: ['country_of_operator_owner', 'operator_owner', 'purpose', 'detailed_purpose', 'launch_mass_kg', 'dry_mass_kg', 'power_watts', 'date_of_launch', 'launch_site', 'launch_vehicle', 'cospar_number', 'norad_number']
OWID posibles claves: ['entity', 'code', 'year', 'annual_launches']


In [ ]:
import numpy as np

satcat_qc = satcat.copy()
ucs_qc = ucs.copy()
owid_qc = owid.copy()

print("=== TIPOS DE DATOS DE LAS CLAVES ===")
print("SATCAT norad_cat_id:", satcat_qc["norad_cat_id"].dtype)
print("SATCAT object_id:", satcat_qc["object_id"].dtype)
print("UCS norad_number:", ucs_qc["norad_number"].dtype)
print("UCS cospar_number:", ucs_qc["cospar_number"].dtype)

print("\n=== NULOS EN CLAVES ===")
print("SATCAT norad_cat_id nulos:", satcat_qc["norad_cat_id"].isna().sum())
print("SATCAT object_id nulos:", satcat_qc["object_id"].isna().sum())
print("UCS norad_number nulos:", ucs_qc["norad_number"].isna().sum())
print("UCS cospar_number nulos:", ucs_qc["cospar_number"].isna().sum())

print("\n=== DUPLICADOS EN CLAVES ===")
print("SATCAT norad_cat_id duplicados:", satcat_qc["norad_cat_id"].duplicated().sum())
print("SATCAT object_id duplicados:", satcat_qc["object_id"].duplicated().sum())
print("UCS norad_number duplicados:", ucs_qc["norad_number"].duplicated().sum())
print("UCS cospar_number duplicados:", ucs_qc["cospar_number"].duplicated().sum())

=== TIPOS DE DATOS DE LAS CLAVES ===
SATCAT norad_cat_id: int64
SATCAT object_id: object
UCS norad_number: int64
UCS cospar_number: object

=== NULOS EN CLAVES ===
SATCAT norad_cat_id nulos: 0
SATCAT object_id nulos: 0
UCS norad_number nulos: 0
UCS cospar_number nulos: 0

=== DUPLICADOS EN CLAVES ===
SATCAT norad_cat_id duplicados: 0
SATCAT object_id duplicados: 0
UCS norad_number duplicados: 9
UCS cospar_number duplicados: 9


In [ ]:
# Normalizo mínimamente las claves solo para comparar
satcat_keys = (
    satcat_qc["norad_cat_id"]
    .dropna()
    .astype("Int64")
)

ucs_keys = (
    ucs_qc["norad_number"]
    .dropna()
    .astype("Int64")
)

satcat_key_set = set(satcat_keys.tolist())
ucs_key_set = set(ucs_keys.tolist())

intersection = satcat_key_set & ucs_key_set
only_satcat = satcat_key_set - ucs_key_set
only_ucs = ucs_key_set - satcat_key_set

print("=== SOLAPAMIENTO DE CLAVES ===")
print("Claves SATCAT:", len(satcat_key_set))
print("Claves UCS:", len(ucs_key_set))
print("En ambas:", len(intersection))
print("Solo SATCAT:", len(only_satcat))
print("Solo UCS:", len(only_ucs))

print("\nCobertura UCS sobre SATCAT: {:.2%}".format(len(intersection) / len(satcat_key_set)))
print("Cobertura SATCAT sobre UCS: {:.2%}".format(len(intersection) / len(ucs_key_set)))

=== SOLAPAMIENTO DE CLAVES ===
Claves SATCAT: 68590
Claves UCS: 7551
En ambas: 7551
Solo SATCAT: 61039
Solo UCS: 0

Cobertura UCS sobre SATCAT: 11.01%
Cobertura SATCAT sobre UCS: 100.00%


In [11]:
satcat_cov = satcat_qc.copy()
satcat_cov["norad_cat_id"] = satcat_cov["norad_cat_id"].astype("Int64")
satcat_cov["in_ucs"] = satcat_cov["norad_cat_id"].isin(ucs_key_set)

coverage_by_type = (
    satcat_cov
    .groupby("object_type", dropna=False)
    .agg(
        n=("norad_cat_id", "size"),
        matched_in_ucs=("in_ucs", "sum")
    )
    .assign(match_rate=lambda df: df["matched_in_ucs"] / df["n"])
    .sort_values("n", ascending=False)
)

print(coverage_by_type)

                 n  matched_in_ucs  match_rate
object_type                                   
DEB          35751              15     0.00042
PAY          25835            7496    0.290149
R/B           6833               6    0.000878
UNK            171              34     0.19883


In [10]:
ucs_non_null = (
    ucs_qc.notna()
    .sum()
    .to_frame("non_null_count")
    .assign(
        total_rows=len(ucs_qc),
        non_null_pct=lambda df: df["non_null_count"] / df["total_rows"]
    )
    .sort_values("non_null_pct", ascending=False)
)

print(ucs_non_null.head(30))

                                    non_null_count  total_rows  non_null_pct
name_of_satellite_alternate_names             7560        7560      1.000000
current_official_name_of_satellite            7560        7560      1.000000
country_of_operator_owner                     7560        7560      1.000000
operator_owner                                7560        7560      1.000000
cospar_number                                 7560        7560      1.000000
users                                         7560        7560      1.000000
purpose                                       7560        7560      1.000000
class_of_orbit                                7560        7560      1.000000
launch_vehicle                                7560        7560      1.000000
launch_site                                   7560        7560      1.000000
country_of_contractor                         7560        7560      1.000000
contractor                                    7560        7560      1.000000

In [9]:
suspect_cols = [c for c in ucs_qc.columns if c.startswith("unnamed") or c.startswith("source")]
print(ucs_non_null.loc[suspect_cols].sort_values("non_null_pct", ascending=False))

                              non_null_count  total_rows  non_null_pct
source_used_for_orbital_data            6636        7560      0.877778
source                                  3286        7560      0.434656
source_2                                1832        7560      0.242328
source_3                                1126        7560      0.148942
source_4                                 729        7560      0.096429
source_1                                 725        7560      0.095899
source_5                                 553        7560      0.073148
source_6                                 504        7560      0.066667
unnamed_60                               488        7560      0.064550
unnamed_61                               488        7560      0.064550
unnamed_62                               487        7560      0.064418
unnamed_66                               487        7560      0.064418
unnamed_56                               485        7560      0.064153
unname

In [4]:
from pathlib import Path

flourish_dir = Path("../data/flourish")

for path in sorted(flourish_dir.glob("*.csv")):
    df = pd.read_csv(path)
    print("\n" + "=" * 100)
    print(path.name)
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print(df.head())


00_kpi_intro.csv
shape: (4, 5)
columns: ['metric_id', 'label', 'value', 'unit', 'description']
           metric_id                             label  value     unit  \
0      total_objects               Objetos registrados  68590  objetos   
1   objects_in_orbit  Objetos sin fecha de decaimiento  33576  objetos   
2  payloads_in_orbit           Cargas útiles en órbita  18572  objetos   
3    debris_in_orbit                Residuos en órbita  12541  objetos   

                                         description  
0  Total de objetos orbitales registrados en SATCAT.  
1  Objetos que permanecen en órbita según la ause...  
2  Satélites o cargas útiles sin fecha de decaimi...  
3  Fragmentos o debris sin fecha de decaimiento r...  

01_owid_country_year.csv
shape: (1131, 5)
columns: ['entity', 'code', 'year', 'annual_launches', 'cumulative_launches']
    entity code  year  annual_launches  cumulative_launches
0  Algeria  DZA  2002                1                    1
1  Algeria  DZA  